In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir('/content/drive/MyDrive/OULAD_processed')

In [ ]:
import pandas as pd
OUTPUT_PATH = '/content/drive/MyDrive/OULAD_processed/'

In [ ]:
!pip install optuna xgboost shap --quiet

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score)
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report, confusion_matrix, average_precision_score, precision_recall_curve, brier_score_loss)

print("=" * 60)
print("XGBoost + Optuna — Modèle Statique Jour 0")
print("=" * 60)


# ════════════════════════════════════════════════════════════
# ÉTAPE 1 — DONNÉES + FEATURES V2
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 1 — Préparation des données")
print("=" * 60)

final_static_df = pd.read_csv(OUTPUT_PATH + 'oulad_final_static.csv')

static_features = [
    'num_of_prev_attempts',
    'studied_credits',
    'imd_score',
    'gender_M',
    'disability_Y',
    'highest_education_num',
    'age_band_num',
    'module_presentation_length',
    'registration_lead_time',
    'registration_missing',
    'module_AAA',
    'module_BBB',
    'module_CCC',
    'module_DDD',
    'module_EEE',
    'module_FFF',
    'module_GGG',
    'presentation_year',
    'presentation_semester',
]

# Features dérivées Jour 0

final_static_df['is_retaker'] = (
    final_static_df['num_of_prev_attempts'] > 0
).astype(int)
final_static_df['credits_per_week'] = (
    final_static_df['studied_credits'] /
    (final_static_df['module_presentation_length'] / 7)
)
final_static_df['very_late_registration'] = (
    final_static_df['registration_lead_time'] < 14
).astype(int)

static_features_v2 = static_features + [
    'is_retaker',
    'credits_per_week',
    'very_late_registration',
]
static_features_v2 = [f for f in static_features_v2
                       if f in final_static_df.columns]

X_static_v2 = final_static_df[static_features_v2].fillna(0).astype(float).values
y_static_v2 = final_static_df['at_risk'].values

print(f"Features v2      : {len(static_features_v2)}")
print(f"Shape X          : {X_static_v2.shape}")
print(f"Taux at_risk     : {y_static_v2.mean()*100:.1f}%")
print(f"Nouvelles features : "
      f"{[f for f in static_features_v2 if f not in static_features]}")

# Split Train / Val / Test
X_tr, X_te, y_tr, y_te = train_test_split(
    X_static_v2, y_static_v2,
    test_size=0.15, stratify=y_static_v2,
    random_state=SEED
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr,
    test_size=0.15, stratify=y_tr,
    random_state=SEED
)

scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()

print(f"\nTrain : {len(X_tr):,} | Val : {len(X_val):,} | "
      f"Test : {len(X_te):,}")
print(f"scale_pos_weight : {scale_pos_weight:.3f}")

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 2 — OPTIMISATION OPTUNA
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 2 — Optimisation Optuna (XGBoost)")
print("=" * 60)

N_TRIALS = 100
N_CV     = 5

cv_inner = StratifiedKFold(
    n_splits=N_CV, shuffle=True, random_state=SEED
)

def objective_xgb(trial):
    params = {
        'n_estimators'      : trial.suggest_int(
                                  'n_estimators', 100, 1000, step=50),
        'learning_rate'     : trial.suggest_float(
                                  'learning_rate', 0.005, 0.3, log=True),
        'max_depth'         : trial.suggest_int('max_depth', 3, 10),
        'min_child_weight'  : trial.suggest_int(
                                  'min_child_weight', 1, 20),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float(
                                  'colsample_bytree', 0.5, 1.0),
        'colsample_bylevel' : trial.suggest_float(
                                  'colsample_bylevel', 0.5, 1.0),
        'reg_alpha'         : trial.suggest_float(
                                  'reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float(
                                  'reg_lambda', 1e-8, 10.0, log=True),
        'gamma'             : trial.suggest_float('gamma', 0.0, 5.0),
        'scale_pos_weight'  : scale_pos_weight,
        'random_state'      : SEED,
        'verbosity'         : 0,
        'eval_metric'       : 'auc',
    }
    model  = xgb.XGBClassifier(**params)
    scores = cross_val_score(
        model, X_tr, y_tr,
        cv=cv_inner, scoring='roc_auc', n_jobs=-1
    )
    return scores.mean()

study = optuna.create_study(
    direction  = 'maximize',
    sampler    = TPESampler(seed=SEED),
    study_name = 'XGBoost_Jour0'
)
study.optimize(
    objective_xgb,
    n_trials          = N_TRIALS,
    timeout           = 1800,
    show_progress_bar = True
)

best_params = study.best_params
print(f"\nMeilleur CV AUC Optuna : {study.best_value:.4f}")
print(f"Meilleurs hyperparamètres :")
for k, v in best_params.items():
    print(f"  {k:<25} : {v}")


In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 3 — ENTRAÎNEMENT MODÈLE FINAL
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 3 — Entraînement modèle final")
print("=" * 60)

final_params = best_params.copy()
final_params.update({
    'scale_pos_weight'    : scale_pos_weight,
    'random_state'        : SEED,
    'verbosity'           : 0,
    'eval_metric'         : 'auc',
    'early_stopping_rounds': 50,
})

xgb_final = xgb.XGBClassifier(**final_params)
xgb_final.fit(
    X_tr, y_tr,
    eval_set = [(X_val, y_val)],
    verbose  = False
)

print(f"Meilleure itération : {xgb_final.best_iteration}")
print(f"Modèle entraîné")


# ════════════════════════════════════════════════════════════
# ETAPE 4 — ÉVALUATION SUR LE TEST SET
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 4 — Évaluation Test Set")
print("=" * 60)

y_proba  = xgb_final.predict_proba(X_te)[:, 1]
y_labels = (y_proba >= 0.5).astype(int)

auc  = roc_auc_score(y_te, y_proba)
ap   = average_precision_score(y_te, y_proba)
brier= brier_score_loss(y_te, y_proba)

tp = ((y_labels == 1) & (y_te == 1)).sum()
fp = ((y_labels == 1) & (y_te == 0)).sum()
fn = ((y_labels == 0) & (y_te == 1)).sum()
tn = ((y_labels == 0) & (y_te == 0)).sum()

prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)
spec = tn / (tn + fp + 1e-8)

print(f"\nRésultats XGBoost + Optuna (Test Set) :")
print(f"  AUC              : {auc:.4f}")
print(f"  AP (Avg Prec.)   : {ap:.4f}")
print(f"  Brier Score      : {brier:.4f}")
print(f"  Precision        : {prec:.4f}")
print(f"  Recall (Sens.)   : {rec:.4f}")
print(f"  Specificity      : {spec:.4f}")
print(f"  F1               : {f1:.4f}")
print(f"\nRapport complet :")
print(classification_report(
    y_te, y_labels,
    target_names=['Non à risque', 'À risque']
))


In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 5 — VISUALISATIONS ÉVALUATION
# ════════════════════════════════════════════════════════════
print("\nETAPE 5 — Visualisations")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'XGBoost + Optuna — Modèle Statique Jour 0',
    fontweight='bold', fontsize=13
)

# ── Courbe ROC ───────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_te, y_proba)
axes[0].plot(fpr, tpr, color='#2563EB', lw=2,
             label=f'XGBoost (AUC={auc:.4f})')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#2563EB')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4,
             label='Aléatoire (AUC=0.5)')
axes[0].set_xlabel('Taux Faux Positifs')
axes[0].set_ylabel('Taux Vrais Positifs')
axes[0].set_title('Courbe ROC', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Courbe Précision-Rappel ───────────────────────────────────
prec_curve, rec_curve, _ = precision_recall_curve(y_te, y_proba)
axes[1].plot(rec_curve, prec_curve, color='#7C3AED', lw=2,
             label=f'XGBoost (AP={ap:.4f})')
axes[1].fill_between(rec_curve, prec_curve,
                      alpha=0.1, color='#7C3AED')
axes[1].axhline(y=y_te.mean(), color='k', ls='--',
                alpha=0.4, label=f'Baseline ({y_te.mean():.2f})')
axes[1].set_xlabel('Rappel')
axes[1].set_ylabel('Précision')
axes[1].set_title('Courbe Précision-Rappel', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# ── Matrice de confusion ──────────────────────────────────────
cm = confusion_matrix(y_te, y_labels)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
    xticklabels=['Non à risque', 'À risque'],
    yticklabels=['Non à risque', 'À risque']
)
axes[2].set_title(f'Matrice de confusion\n'
                   f'AUC={auc:.4f} | F1={f1:.4f}',
                   fontweight='bold')
axes[2].set_ylabel('Réel')
axes[2].set_xlabel('Prédit')

plt.tight_layout()
plt.show()

# ── Distribution des scores de risque ────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_proba[y_te == 0], bins=50, alpha=0.6,
        color='#16A34A', label='Non à risque', density=True)
ax.hist(y_proba[y_te == 1], bins=50, alpha=0.6,
        color='#DC2626', label='À risque', density=True)
ax.axvline(x=0.5, color='black', ls='--',
           lw=2, label='Seuil = 0.5')
ax.set_xlabel('Score de risque prédit')
ax.set_ylabel('Densité')
ax.set_title('Distribution des scores de risque — XGBoost Jour 0',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Historique Optuna ─────────────────────────────────────────
trials_df   = study.trials_dataframe()
best_so_far = trials_df['value'].cummax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Historique Optuna — XGBoost",
             fontweight='bold', fontsize=13)

axes[0].scatter(trials_df.index, trials_df['value'],
                alpha=0.4, s=15, color='#2563EB',
                label='Trial AUC')
axes[0].plot(trials_df.index, best_so_far,
             color='#DC2626', lw=2,
             label=f'Best : {study.best_value:.4f}')
axes[0].axhline(y=study.best_value, color='red',
                ls='--', alpha=0.5)
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('CV AUC')
axes[0].set_title('Évolution CV AUC')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Importance des hyperparamètres
param_importance = optuna.importance.get_param_importances(study)
params_names = list(param_importance.keys())
params_vals  = list(param_importance.values())

axes[1].barh(params_names, params_vals, color='#7C3AED',
             alpha=0.85)
axes[1].set_xlabel('Importance relative')
axes[1].set_title('Importance des hyperparamètres (Optuna)')
axes[1].grid(alpha=0.3, axis='x')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 6 — CROSS-VALIDATION FINALE
# Avec les meilleurs hyperparamètres
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 6 — Cross-Validation finale (5 folds)")
print("=" * 60)

cv_outer = StratifiedKFold(
    n_splits=N_CV, shuffle=True, random_state=SEED
)

cv_params = best_params.copy()
cv_params.update({
    'scale_pos_weight': scale_pos_weight,
    'random_state'    : SEED,
    'verbosity'       : 0,
    'eval_metric'     : 'auc',
})

cv_scores  = []
cv_details = []

for fold, (tr_idx, val_idx) in enumerate(
    cv_outer.split(X_static_v2, y_static_v2)
):
    X_f_tr,  X_f_val  = X_static_v2[tr_idx], X_static_v2[val_idx]
    y_f_tr,  y_f_val  = y_static_v2[tr_idx],  y_static_v2[val_idx]

    spw = (y_f_tr == 0).sum() / (y_f_tr == 1).sum()
    cv_params['scale_pos_weight'] = spw

    model_cv = xgb.XGBClassifier(**cv_params)
    model_cv.fit(X_f_tr, y_f_tr, verbose=False)

    proba_cv = model_cv.predict_proba(X_f_val)[:, 1]
    label_cv = (proba_cv >= 0.5).astype(int)

    fold_auc = roc_auc_score(y_f_val, proba_cv)
    fold_ap  = average_precision_score(y_f_val, proba_cv)

    tp_f = ((label_cv==1) & (y_f_val==1)).sum()
    fp_f = ((label_cv==1) & (y_f_val==0)).sum()
    fn_f = ((label_cv==0) & (y_f_val==1)).sum()
    prec_f = tp_f / (tp_f + fp_f + 1e-8)
    rec_f  = tp_f / (tp_f + fn_f + 1e-8)
    f1_f   = 2*prec_f*rec_f / (prec_f+rec_f+1e-8)

    cv_scores.append(fold_auc)
    cv_details.append({
        'Fold'     : fold + 1,
        'AUC'      : fold_auc,
        'AP'       : fold_ap,
        'Precision': prec_f,
        'Recall'   : rec_f,
        'F1'       : f1_f,
    })

    print(f"  Fold {fold+1} | AUC={fold_auc:.4f} | "
          f"AP={fold_ap:.4f} | F1={f1_f:.4f}")

cv_df = pd.DataFrame(cv_details)

print(f"\nRésumé CV :")
print(f"  AUC  : {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"  AP   : {cv_df['AP'].mean():.4f} ± {cv_df['AP'].std():.4f}")
print(f"  F1   : {cv_df['F1'].mean():.4f} ± {cv_df['F1'].std():.4f}")

# Graphique CV
fig, ax = plt.subplots(figsize=(10, 5))
metrics_cv = ['AUC', 'AP', 'Precision', 'Recall', 'F1']
colors_cv  = ['#7C3AED','#2563EB','#16A34A','#D97706','#DC2626']

x     = np.arange(N_CV)
width = 0.15

for j, (metric, color) in enumerate(zip(metrics_cv, colors_cv)):
    vals = cv_df[metric].tolist()
    ax.bar(x + j*width, vals, width,
           label=metric, color=color, alpha=0.85)

ax.axhline(y=np.mean(cv_scores), color='black',
           ls='--', alpha=0.5,
           label=f'AUC mean={np.mean(cv_scores):.4f}')
ax.set_xticks(x + width*2)
ax.set_xticklabels([f'Fold {i+1}' for i in range(N_CV)])
ax.set_ylabel('Score')
ax.set_title('Cross-Validation — XGBoost + Optuna',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# RESUMÉ FINAL
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("RESUMÉ FINAL — XGBoost + Optuna (Jour 0)")
print("=" * 60)

print(f"\nFeatures          : {len(static_features_v2)} (v2)")
print(f"Optuna trials     : {N_TRIALS}")
print(f"Meilleur CV Optuna: {study.best_value:.4f}")
print(f"\nTest Set :")
print(f"  AUC             : {auc:.4f}")
print(f"  AP              : {ap:.4f}")
print(f"  Brier Score     : {brier:.4f}")
print(f"  Precision       : {prec:.4f}")
print(f"  Recall          : {rec:.4f}")
print(f"  F1              : {f1:.4f}")
print(f"\nCross-Validation (5 folds) :")
print(f"  AUC             : {np.mean(cv_scores):.4f}"
      f" ± {np.std(cv_scores):.4f}")
print(f"  F1              : {cv_df['F1'].mean():.4f}"
      f" ± {cv_df['F1'].std():.4f}")
print(f"\nTop 3 features SHAP :")
for i, feat in enumerate(top3_features):
    print(f"  {i+1}. {feat} "
          f"(SHAP mean={mean_abs_shap[top3_idx[i]]:.4f})")
print(f"\n Modèle statique Jour 0 finalisé")
print(f"→ Prêt pour M4 (SHAP global) et M5 (Personnalisation)")

In [ ]:
# ════════════════════════════════════════════════════════════
# SAUVEGARDE DU MODELE STATIQUE FINAL — XGBoost + Optuna
# ════════════════════════════════════════════════════════════

import os
import pickle
import json
import joblib

SAVE_PATH = OUTPUT_PATH + "models_static/"
os.makedirs(SAVE_PATH, exist_ok=True)

# 1. Sauvegarder le modèle XGBoost final
xgb_final.save_model(SAVE_PATH + "xgb_optuna_static_day0.json")

# 2. Sauvegarder avec joblib aussi
joblib.dump(xgb_final, SAVE_PATH + "xgb_optuna_static_day0.pkl")

# 3. Sauvegarder la liste exacte des features
with open(SAVE_PATH + "static_features_v2.pkl", "wb") as f:
    pickle.dump(static_features_v2, f)

# 4. Sauvegarder les hyperparamètres Optuna
with open(SAVE_PATH + "xgb_optuna_best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

# 5. Sauvegarder les métriques finales
metrics_static = {
    "auc": float(auc),
    "ap": float(ap),
    "brier": float(brier),
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1),
    "cv_auc_mean": float(np.mean(cv_scores)),
    "cv_auc_std": float(np.std(cv_scores)),
    "n_features": len(static_features_v2),
    "best_optuna_cv_auc": float(study.best_value),
}

with open(SAVE_PATH + "xgb_optuna_static_metrics.json", "w") as f:
    json.dump(metrics_static, f, indent=4)

print("Modèle statique XGBoost + Optuna sauvegardé")
print("Dossier :", SAVE_PATH)

In [ ]:
# Sauvegarder les SHAP values
np.save(OUTPUT_PATH + 'shap_values_static.npy', shap_vals)

In [ ]:
import joblib
import pickle

SAVE_PATH = OUTPUT_PATH + "models_static/"

xgb_loaded = joblib.load(SAVE_PATH + "xgb_optuna_static_day0.pkl")

with open(SAVE_PATH + "static_features_v2.pkl", "rb") as f:
    static_features_v2_loaded = pickle.load(f)

print("Modèle rechargé ")
print("Nombre de features :", len(static_features_v2_loaded))

In [ ]:
# ════════════════════════════════════════════════════════════
# RECHERCHE DU SEUIL OPTIMAL — XGBOOST JOUR 0
# ════════════════════════════════════════════════════════════

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("="*60)
print("RECHERCHE DU SEUIL OPTIMAL — XGBOOST")
print("="*60)

SEED = 42

# ----------------------------------------------------------
# 1. Charger les features sauvegardées
# ----------------------------------------------------------

import pickle

with open(
    OUTPUT_PATH + "models_static/static_features_v2.pkl",
    "rb"
) as f:
    static_features_v2 = pickle.load(f)

# ----------------------------------------------------------
# 2. Recharger le dataset
# ----------------------------------------------------------

final_static_df = pd.read_csv(
    OUTPUT_PATH + "oulad_final_static.csv"
)

# ----------------------------------------------------------
# 3. Recréer les features dérivées
# ----------------------------------------------------------

final_static_df['is_retaker'] = (
    final_static_df['num_of_prev_attempts'] > 0
).astype(int)

final_static_df['credits_per_week'] = (
    final_static_df['studied_credits']
    / (final_static_df['module_presentation_length'] / 7)
)

final_static_df['very_late_registration'] = (
    final_static_df['registration_lead_time'] < 14
).astype(int)

# ----------------------------------------------------------
# 4. Reconstruction X et y
# ----------------------------------------------------------

X_static_v2 = (
    final_static_df[static_features_v2]
    .fillna(0)
    .astype(float)
    .values
)

y_static_v2 = final_static_df['at_risk'].values

# ----------------------------------------------------------
# 5. Refaire exactement le split original
# ----------------------------------------------------------

X_tr, X_te, y_tr, y_te = train_test_split(
    X_static_v2,
    y_static_v2,
    test_size=0.15,
    stratify=y_static_v2,
    random_state=SEED
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr,
    y_tr,
    test_size=0.15,
    stratify=y_tr,
    random_state=SEED
)

print(f"Test set : {len(X_te)} étudiants")

# ----------------------------------------------------------
# 6. Charger le modèle sauvegardé
# ----------------------------------------------------------

model = joblib.load(
    OUTPUT_PATH +
    "models_static/xgb_optuna_static_day0.pkl"
)

# ----------------------------------------------------------
# 7. Calcul des probabilités
# ----------------------------------------------------------

y_proba = model.predict_proba(X_te)[:,1]

print("\nProbabilités calculées.")

In [ ]:

# ----------------------------------------------------------
# 8. Tester tous les seuils
# ----------------------------------------------------------

thresholds = np.arange(0.30, 0.71, 0.05)

results = []

for th in thresholds:

    y_pred = (y_proba >= th).astype(int)

    precision = precision_score(
        y_te,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_te,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_te,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_te,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp + 1e-8)

    results.append([
        th,
        precision,
        recall,
        specificity,
        f1
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "threshold",
        "precision",
        "recall",
        "specificity",
        "f1"
    ]
)

# ----------------------------------------------------------
# 9. Meilleur seuil F1
# ----------------------------------------------------------

best_f1_row = results_df.loc[
    results_df["f1"].idxmax()
]

best_f1_threshold = best_f1_row["threshold"]

# ----------------------------------------------------------
# 10. Meilleur seuil Recall
# ----------------------------------------------------------

best_recall_row = results_df.loc[
    results_df["recall"].idxmax()
]

best_recall_threshold = best_recall_row["threshold"]

# ----------------------------------------------------------
# 11. Résultats seuil 0.50
# ----------------------------------------------------------

row_05 = results_df.iloc[
    (results_df["threshold"] - 0.50)
    .abs()
    .argmin()
]

# ----------------------------------------------------------
# 12. Affichage
# ----------------------------------------------------------

print("\n" + "="*60)
print("SEUIL STANDARD (0.50)")
print("="*60)

print(row_05)

print("\n" + "="*60)
print("MEILLEUR SEUIL F1")
print("="*60)

print(best_f1_row)

print("\n" + "="*60)
print("MEILLEUR SEUIL RECALL")
print("="*60)

print(best_recall_row)

# ----------------------------------------------------------
# 13. Tableau comparatif
# ----------------------------------------------------------

summary = pd.DataFrame([
    row_05,
    best_f1_row,
    best_recall_row
])

summary.index = [
    "Seuil 0.50",
    "Optimal F1",
    "Optimal Recall"
]

print("\n")
print(summary.round(4))

# ----------------------------------------------------------
# 14. Courbes
# ----------------------------------------------------------

plt.figure(figsize=(12,6))

plt.plot(
    results_df["threshold"],
    results_df["precision"],
    label="Precision"
)

plt.plot(
    results_df["threshold"],
    results_df["recall"],
    label="Recall"
)

plt.plot(
    results_df["threshold"],
    results_df["f1"],
    label="F1"
)

plt.axvline(
    best_f1_threshold,
    color='red',
    linestyle='--',
    label=f'Best F1 = {best_f1_threshold:.2f}'
)

plt.axvline(
    0.50,
    color='black',
    linestyle=':',
    label='Threshold = 0.50'
)

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Impact du seuil de classification")
plt.grid(alpha=0.3)
plt.legend()

plt.show()

In [ ]:
import joblib

xgb_final = joblib.load(
    OUTPUT_PATH + "models_static/xgb_optuna_static_day0.pkl"
)

print("Modèle chargé")

In [ ]:
y_proba = xgb_final.predict_proba(X_te)[:, 1]

In [ ]:
xgb_predictions = pd.DataFrame({
    "y_true": y_te,
    "risk_score": y_proba
})

xgb_predictions.to_csv(
    OUTPUT_PATH + "xgb_predictions.csv",
    index=False
)

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 4 — EVALUATION SUR LE TEST SET
# SEUIL OPTIMAL = 0.35
# ════════════════════════════════════════════════════════════
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report
)

print("\nÉTAPE 4 — Évaluation Test Set")
print("=" * 60)

OPTIMAL_THRESHOLD = 0.35

# Probabilités
y_proba = xgb_final.predict_proba(X_te)[:, 1]

# Classification avec seuil optimal
y_labels = (y_proba >= OPTIMAL_THRESHOLD).astype(int)

# Métriques indépendantes du seuil
auc   = roc_auc_score(y_te, y_proba)
ap    = average_precision_score(y_te, y_proba)
brier = brier_score_loss(y_te, y_proba)

# Matrice de confusion
tp = ((y_labels == 1) & (y_te == 1)).sum()
fp = ((y_labels == 1) & (y_te == 0)).sum()
fn = ((y_labels == 0) & (y_te == 1)).sum()
tn = ((y_labels == 0) & (y_te == 0)).sum()

# Métriques dépendantes du seuil
prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)
spec = tn / (tn + fp + 1e-8)

print(f"\nRésultats XGBoost + Optuna (Test Set)")
print(f"Seuil utilisé : {OPTIMAL_THRESHOLD:.2f}")

print(f"\nAUC              : {auc:.4f}")
print(f"AP               : {ap:.4f}")
print(f"Brier Score      : {brier:.4f}")
print(f"Precision        : {prec:.4f}")
print(f"Recall           : {rec:.4f}")
print(f"Specificity      : {spec:.4f}")
print(f"F1-score         : {f1:.4f}")

print("\nRapport complet :")
print(
    classification_report(
        y_te,
        y_labels,
        target_names=['Non à risque', 'À risque']
    )
)

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 5 — VISUALISATIONS ÉVALUATION
# ════════════════════════════════════════════════════════════
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    precision_recall_curve
)

print("\nÉTAPE 5 — Visualisations")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'XGBoost + Optuna — Modèle Statique Jour 0',
    fontweight='bold', fontsize=13
)

# ── Courbe ROC ───────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_te, y_proba)
axes[0].plot(fpr, tpr, color='#2563EB', lw=2,
             label=f'XGBoost (AUC={auc:.4f})')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#2563EB')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4,
             label='Aléatoire (AUC=0.5)')
axes[0].set_xlabel('Taux Faux Positifs')
axes[0].set_ylabel('Taux Vrais Positifs')
axes[0].set_title('Courbe ROC', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Courbe Précision-Rappel ───────────────────────────────────
prec_curve, rec_curve, _ = precision_recall_curve(y_te, y_proba)
axes[1].plot(rec_curve, prec_curve, color='#7C3AED', lw=2,
             label=f'XGBoost (AP={ap:.4f})')
axes[1].fill_between(rec_curve, prec_curve,
                      alpha=0.1, color='#7C3AED')
axes[1].axhline(y=y_te.mean(), color='k', ls='--',
                alpha=0.4, label=f'Baseline ({y_te.mean():.2f})')
axes[1].set_xlabel('Rappel')
axes[1].set_ylabel('Précision')
axes[1].set_title('Courbe Précision-Rappel', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# ── Matrice de confusion ──────────────────────────────────────
cm = confusion_matrix(y_te, y_labels)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
    xticklabels=['Non à risque', 'À risque'],
    yticklabels=['Non à risque', 'À risque']
)
axes[2].set_title(f'Matrice de confusion\n'
                   f'AUC={auc:.4f} | F1={f1:.4f}',
                   fontweight='bold')
axes[2].set_ylabel('Réel')
axes[2].set_xlabel('Prédit')

plt.tight_layout()
plt.show()

# ── Distribution des scores de risque ────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_proba[y_te == 0], bins=50, alpha=0.6,
        color='#16A34A', label='Non à risque', density=True)
ax.hist(y_proba[y_te == 1], bins=50, alpha=0.6,
        color='#DC2626', label='À risque', density=True)
ax.axvline(x=OPTIMAL_THRESHOLD, color='black', ls='--',
           lw=2, label=f'Seuil optimal = {OPTIMAL_THRESHOLD:.2f}')
ax.set_xlabel('Score de risque prédit')
ax.set_ylabel('Densité')
ax.set_title('Distribution des scores de risque — XGBoost Jour 0',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Historique Optuna ─────────────────────────────────────────
trials_df   = study.trials_dataframe()
best_so_far = trials_df['value'].cummax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Historique Optuna — XGBoost",
             fontweight='bold', fontsize=13)

axes[0].scatter(trials_df.index, trials_df['value'],
                alpha=0.4, s=15, color='#2563EB',
                label='Trial AUC')
axes[0].plot(trials_df.index, best_so_far,
             color='#DC2626', lw=2,
             label=f'Best : {study.best_value:.4f}')
axes[0].axhline(y=study.best_value, color='red',
                ls='--', alpha=0.5)
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('CV AUC')
axes[0].set_title('Évolution CV AUC')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Importance des hyperparamètres
param_importance = optuna.importance.get_param_importances(study)
params_names = list(param_importance.keys())
params_vals  = list(param_importance.values())

axes[1].barh(params_names, params_vals, color='#7C3AED',
             alpha=0.85)
axes[1].set_xlabel('Importance relative')
axes[1].set_title('Importance des hyperparamètres (Optuna)')
axes[1].grid(alpha=0.3, axis='x')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════
# XGBoost + Optuna — Modèle Statique Jour 0
# Test : booster gbtree vs dart
# ════════════════════════════════════════════════════════════

In [ ]:
!pip install optuna xgboost shap --quiet

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split,
                                     StratifiedKFold,
                                     cross_val_score)
from sklearn.metrics import (roc_auc_score, roc_curve,
                              classification_report,
                              confusion_matrix,
                              average_precision_score,
                              precision_recall_curve,
                              brier_score_loss)

print("=" * 60)
print("XGBoost + Optuna — Modèle Statique Jour 0")
print("Boosters testés : gbtree vs dart")
print("=" * 60)


# ════════════════════════════════════════════════════════════
# ETAPE 1 — DONNEES + FEATURES V2
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 1 — Préparation des données")
print("=" * 60)

final_static_df = pd.read_csv(OUTPUT_PATH + 'oulad_final_static.csv')

static_features = [
    'num_of_prev_attempts', 'studied_credits', 'imd_score',
    'gender_M', 'disability_Y', 'highest_education_num',
    'age_band_num', 'module_presentation_length',
    'registration_lead_time', 'registration_missing',
    'module_AAA', 'module_BBB', 'module_CCC', 'module_DDD',
    'module_EEE', 'module_FFF', 'module_GGG',
    'presentation_year', 'presentation_semester',
]

# Features dérivées Jour 0
final_static_df['is_retaker'] = (
    final_static_df['num_of_prev_attempts'] > 0
).astype(int)
final_static_df['credits_per_week'] = (
    final_static_df['studied_credits'] /
    (final_static_df['module_presentation_length'] / 7)
)
final_static_df['very_late_registration'] = (
    final_static_df['registration_lead_time'] < 14
).astype(int)

static_features_v2 = static_features + [
    'is_retaker', 'credits_per_week', 'very_late_registration',
]
static_features_v2 = [f for f in static_features_v2
                       if f in final_static_df.columns]

X_static_v2 = final_static_df[static_features_v2].fillna(0).astype(float).values
y_static_v2 = final_static_df['at_risk'].values

print(f"Features v2      : {len(static_features_v2)}")
print(f"Shape X          : {X_static_v2.shape}")
print(f"Taux at_risk     : {y_static_v2.mean()*100:.1f}%")

X_tr, X_te, y_tr, y_te = train_test_split(
    X_static_v2, y_static_v2,
    test_size=0.15, stratify=y_static_v2, random_state=SEED
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr,
    test_size=0.15, stratify=y_tr, random_state=SEED
)

scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f"\nTrain : {len(X_tr):,} | Val : {len(X_val):,} | "
      f"Test : {len(X_te):,}")
print(f"scale_pos_weight : {scale_pos_weight:.3f}")

N_TRIALS = 100
N_CV     = 5
cv_inner = StratifiedKFold(n_splits=N_CV, shuffle=True,
                             random_state=SEED)
BOOSTERS = ['gbtree', 'dart']


In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 2 — OPTIMISATION OPTUNA (gbtree + dart)
# ════════════════════════════════════════════════════════════
print("\nETAPE 2 — Optimisation Optuna")
print("=" * 60)

# gbtree peut être optimisé plus largement
N_TRIALS_GBTREE = 50
TIMEOUT_GBTREE  = 7200

# dart est très lent, donc on limite
N_TRIALS_DART = 15
TIMEOUT_DART  = 10000
def get_params(trial, booster):
    """Génère les hyperparamètres selon le booster."""

    if booster == 'gbtree':
        n_estimators_range = (100, 1000)
        max_depth_range = (3, 10)
        learning_rate_range = (0.005, 0.3)
    else:  # dart
        n_estimators_range = (100, 500)
        max_depth_range = (3, 6)
        learning_rate_range = (0.01, 0.1)

    params = {
        'booster'           : booster,
        'n_estimators'      : trial.suggest_int(
                                  'n_estimators',
                                  n_estimators_range[0],
                                  n_estimators_range[1],
                                  step=50),
        'learning_rate'     : trial.suggest_float(
                                  'learning_rate',
                                  learning_rate_range[0],
                                  learning_rate_range[1],
                                  log=True),
        'max_depth'         : trial.suggest_int(
                                  'max_depth',
                                  max_depth_range[0],
                                  max_depth_range[1]),
        'min_child_weight'  : trial.suggest_int(
                                  'min_child_weight', 1, 10),
        'subsample'         : trial.suggest_float(
                                  'subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float(
                                  'colsample_bytree', 0.6, 1.0),
        'colsample_bylevel' : trial.suggest_float(
                                  'colsample_bylevel', 0.6, 1.0),
        'reg_alpha'         : trial.suggest_float(
                                  'reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda'        : trial.suggest_float(
                                  'reg_lambda', 1e-4, 10.0, log=True),
        'gamma'             : trial.suggest_float(
                                  'gamma', 0.0, 3.0),
        'scale_pos_weight'  : scale_pos_weight,
        'random_state'      : SEED,
        'verbosity'         : 0,
        'eval_metric'       : 'auc',
        'n_jobs'            : 1,
    }

    if booster == 'dart':
        params['rate_drop'] = trial.suggest_float(
            'rate_drop', 0.05, 0.3
        )
        params['skip_drop'] = trial.suggest_float(
            'skip_drop', 0.0, 0.3
        )
        params['sample_type'] = trial.suggest_categorical(
            'sample_type', ['uniform', 'weighted']
        )
        params['normalize_type'] = trial.suggest_categorical(
            'normalize_type', ['tree', 'forest']
        )

    return params

    # Paramètres spécifiques à DART
    if booster == 'dart':
        params['rate_drop']    = trial.suggest_float(
                                     'rate_drop', 0.0, 0.5)
        params['skip_drop']    = trial.suggest_float(
                                     'skip_drop', 0.0, 0.5)
        params['sample_type']  = trial.suggest_categorical(
                                     'sample_type',
                                     ['uniform', 'weighted'])
        params['normalize_type'] = trial.suggest_categorical(
                                     'normalize_type',
                                     ['tree', 'forest'])

    return params

In [ ]:
studies      = {}
best_results = {}

for booster in BOOSTERS:
    print(f"\nOptimisation — booster : {booster}")

    def objective(trial, b=booster):
        params = get_params(trial, b)
        model  = xgb.XGBClassifier(**params)
        scores = cross_val_score(
            model, X_tr, y_tr,
            cv=cv_inner, scoring='roc_auc', n_jobs=-1
        )
        return scores.mean()

    study = optuna.create_study(
        direction  = 'maximize',
        sampler    = TPESampler(seed=SEED),
        study_name = f'XGBoost_{booster}'
    )


    if booster == 'dart':
      n_trials_current = N_TRIALS_DART
      timeout_current = TIMEOUT_DART
    else:
      n_trials_current = N_TRIALS_GBTREE
      timeout_current = TIMEOUT_GBTREE

    study.optimize(
        objective,
        n_trials=n_trials_current,
        timeout=timeout_current,
        show_progress_bar=True
    )

    studies[booster] = study
    best_results[booster] = {
        'best_cv_auc': study.best_value,
        'best_params': study.best_params,
    }

    print(f"\n{booster.upper()}")
    print(f"  Meilleur CV AUC : {study.best_value:.4f}")
    print(f"  Meilleurs params :")
    for k, v in study.best_params.items():
        print(f"    {k:<25} : {v}")

# Comparaison préliminaire
print(f"\nComparaison CV AUC :")
for booster in BOOSTERS:
    print(f"  {booster:<10} : {best_results[booster]['best_cv_auc']:.4f}")

best_booster = max(BOOSTERS,
                   key=lambda b: best_results[b]['best_cv_auc'])
print(f"\n→ Meilleur booster : {best_booster.upper()} "
      f"(CV AUC={best_results[best_booster]['best_cv_auc']:.4f})")


In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 3 — ENTRAÎNEMENT DES 2 MODELES FINAUX
# ════════════════════════════════════════════════════════════
print("\nETAPE 3 — Entraînement modèles finaux")
print("=" * 60)

trained_models = {}
test_metrics   = {}

for booster in BOOSTERS:
    print(f"\nEntraînement {booster.upper()}...")

    final_params = best_results[booster]['best_params'].copy()
    final_params.update({
        'booster'             : booster,
        'scale_pos_weight'    : scale_pos_weight,
        'random_state'        : SEED,
        'verbosity'           : 0,
        'eval_metric'         : 'auc',
        'use_label_encoder'   : False,
    })

    # DART ne supporte pas early_stopping avec predict_proba
    if booster == 'gbtree':
        final_params['early_stopping_rounds'] = 50
        model = xgb.XGBClassifier(**final_params)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  verbose=False)
        print(f"  Meilleure itération : {model.best_iteration}")
    else:
        model = xgb.XGBClassifier(**final_params)
        model.fit(X_tr, y_tr, verbose=False)
        print(f"  DART entraîné (pas d'early stopping)")

    trained_models[booster] = model

    # Évaluation Test
    y_proba_b  = model.predict_proba(X_te)[:, 1]
    y_labels_b = (y_proba_b >= 0.5).astype(int)

    auc_b  = roc_auc_score(y_te, y_proba_b)
    ap_b   = average_precision_score(y_te, y_proba_b)
    brier_b= brier_score_loss(y_te, y_proba_b)

    tp_b = ((y_labels_b==1) & (y_te==1)).sum()
    fp_b = ((y_labels_b==1) & (y_te==0)).sum()
    fn_b = ((y_labels_b==0) & (y_te==1)).sum()
    tn_b = ((y_labels_b==0) & (y_te==0)).sum()

    prec_b = tp_b / (tp_b + fp_b + 1e-8)
    rec_b  = tp_b / (tp_b + fn_b + 1e-8)
    f1_b   = 2*prec_b*rec_b / (prec_b+rec_b+1e-8)
    spec_b = tn_b / (tn_b + fp_b + 1e-8)

    test_metrics[booster] = {
        'proba'    : y_proba_b,
        'labels'   : y_labels_b,
        'auc'      : auc_b,
        'ap'       : ap_b,
        'brier'    : brier_b,
        'precision': prec_b,
        'recall'   : rec_b,
        'f1'       : f1_b,
        'spec'     : spec_b,
    }

    print(f"  Test AUC  : {auc_b:.4f}")
    print(f"  Test F1   : {f1_b:.4f}")
    print(f"  Brier     : {brier_b:.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 4 — COMPARAISON GBTREE vs DART
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 4 — Comparaison gbtree vs dart")
print("=" * 60)

comparison_df = pd.DataFrame([
    {
        'Booster'         : b.upper(),
        'CV AUC (Optuna)' : best_results[b]['best_cv_auc'],
        'Test AUC'        : test_metrics[b]['auc'],
        'Test AP'         : test_metrics[b]['ap'],
        'Test Precision'  : test_metrics[b]['precision'],
        'Test Recall'     : test_metrics[b]['recall'],
        'Test F1'         : test_metrics[b]['f1'],
        'Brier Score'     : test_metrics[b]['brier'],
    }
    for b in BOOSTERS
]).set_index('Booster')

print("\nTableau comparatif :")
display(comparison_df.round(4))

# Modèle final retenu
best_booster_test = max(BOOSTERS, key=lambda b: test_metrics[b]['auc'])
xgb_final         = trained_models[best_booster_test]
y_proba           = test_metrics[best_booster_test]['proba']
y_labels          = test_metrics[best_booster_test]['labels']
auc               = test_metrics[best_booster_test]['auc']
ap                = test_metrics[best_booster_test]['ap']
brier             = test_metrics[best_booster_test]['brier']
prec              = test_metrics[best_booster_test]['precision']
rec               = test_metrics[best_booster_test]['recall']
f1                = test_metrics[best_booster_test]['f1']

print(f"\n Booster retenu : {best_booster_test.upper()}")
print(f"   Test AUC : {auc:.4f}")
print(f"\nRapport complet :")
print(classification_report(
    y_te, y_labels,
    target_names=['Non à risque', 'À risque']
))


In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 5 — VISUALISATIONS
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 5 — Visualisations")
print("=" * 60)

colors_boosters = {'gbtree': '#2563EB', 'dart': '#DC2626'}

# ── Plot 1 : ROC + PR + Confusion du meilleur modèle ─────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'XGBoost {best_booster_test.upper()} + Optuna — '
    f'Modèle Statique Jour 0',
    fontweight='bold', fontsize=13
)

fpr, tpr, _ = roc_curve(y_te, y_proba)
axes[0].plot(fpr, tpr, color=colors_boosters[best_booster_test],
             lw=2, label=f'{best_booster_test.upper()} '
                         f'(AUC={auc:.4f})')
axes[0].fill_between(fpr, tpr, alpha=0.1,
                      color=colors_boosters[best_booster_test])
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4, label='Aléatoire')
axes[0].set_xlabel('Taux Faux Positifs')
axes[0].set_ylabel('Taux Vrais Positifs')
axes[0].set_title('Courbe ROC', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

prec_c, rec_c, _ = precision_recall_curve(y_te, y_proba)
axes[1].plot(rec_c, prec_c,
             color=colors_boosters[best_booster_test],
             lw=2, label=f'AP={ap:.4f}')
axes[1].fill_between(rec_c, prec_c, alpha=0.1,
                      color=colors_boosters[best_booster_test])
axes[1].axhline(y=y_te.mean(), color='k', ls='--', alpha=0.4,
                label=f'Baseline ({y_te.mean():.2f})')
axes[1].set_xlabel('Rappel')
axes[1].set_ylabel('Précision')
axes[1].set_title('Courbe Précision-Rappel', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

cm = confusion_matrix(y_te, y_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Non à risque', 'À risque'],
            yticklabels=['Non à risque', 'À risque'])
axes[2].set_title(f'Matrice de confusion\n'
                   f'AUC={auc:.4f} | F1={f1:.4f}',
                   fontweight='bold')
axes[2].set_ylabel('Réel')
axes[2].set_xlabel('Prédit')
plt.tight_layout()
plt.show()

# ── Plot 2 : ROC superposées gbtree vs dart ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparaison gbtree vs dart',
             fontweight='bold', fontsize=13)

for booster in BOOSTERS:
    m      = test_metrics[booster]
    fpr_b, tpr_b, _ = roc_curve(y_te, m['proba'])
    axes[0].plot(fpr_b, tpr_b, lw=2,
                 color=colors_boosters[booster],
                 label=f"{booster.upper()} "
                       f"(AUC={m['auc']:.4f})")
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[0].set_xlabel('Taux Faux Positifs')
axes[0].set_ylabel('Taux Vrais Positifs')
axes[0].set_title('Courbes ROC — gbtree vs dart',
                   fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

metrics_to_plot = ['auc', 'ap', 'precision', 'recall', 'f1']
x      = np.arange(len(metrics_to_plot))
width  = 0.3
for j, booster in enumerate(BOOSTERS):
    vals = [test_metrics[booster][m] for m in metrics_to_plot]
    axes[1].bar(x + j*width, vals, width,
                label=booster.upper(),
                color=colors_boosters[booster], alpha=0.85)

axes[1].set_xticks(x + width/2)
axes[1].set_xticklabels(
    ['AUC', 'AP', 'Precision', 'Recall', 'F1'], fontsize=10
)
axes[1].set_ylabel('Score')
axes[1].set_title('Métriques — gbtree vs dart',
                   fontweight='bold')
axes[1].legend()
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# ── Plot 3 : Distribution des scores de risque ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des scores de risque',
             fontweight='bold', fontsize=13)

for ax, booster in zip(axes, BOOSTERS):
    proba_b = test_metrics[booster]['proba']
    ax.hist(proba_b[y_te == 0], bins=50, alpha=0.6,
            color='#16A34A', label='Non à risque', density=True)
    ax.hist(proba_b[y_te == 1], bins=50, alpha=0.6,
            color='#DC2626', label='À risque', density=True)
    ax.axvline(x=0.5, color='black', ls='--', lw=2,
               label='Seuil = 0.5')
    ax.set_xlabel('Score de risque')
    ax.set_ylabel('Densité')
    ax.set_title(f'{booster.upper()} — '
                 f'AUC={test_metrics[booster]["auc"]:.4f}',
                 fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Plot 4 : Historique Optuna ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Historique Optuna — gbtree vs dart",
             fontweight='bold', fontsize=13)

for ax, booster in zip(axes, BOOSTERS):
    trials_df   = studies[booster].trials_dataframe()
    best_so_far = trials_df['value'].cummax()

    ax.scatter(trials_df.index, trials_df['value'],
               alpha=0.4, s=15, color=colors_boosters[booster],
               label='Trial AUC')
    ax.plot(trials_df.index, best_so_far,
            color='black', lw=2,
            label=f"Best : {studies[booster].best_value:.4f}")
    ax.axhline(y=studies[booster].best_value,
               color='red', ls='--', alpha=0.5)
    ax.set_xlabel('Trial')
    ax.set_ylabel('CV AUC')
    ax.set_title(f'Optuna — {booster.upper()}',
                 fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Plot 5 : Importance des hyperparamètres ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Importance des hyperparamètres (Optuna)",
             fontweight='bold', fontsize=13)

for ax, booster in zip(axes, BOOSTERS):
    param_imp = optuna.importance.get_param_importances(
        studies[booster]
    )
    names = list(param_imp.keys())
    vals  = list(param_imp.values())
    ax.barh(names, vals, color=colors_boosters[booster], alpha=0.85)
    ax.set_xlabel('Importance relative')
    ax.set_title(f'{booster.upper()}', fontweight='bold')
    ax.grid(alpha=0.3, axis='x')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 6 — CROSS-VALIDATION FINALE (meilleur booster)
# ════════════════════════════════════════════════════════════
print("\nETAPE 6 — Cross-Validation finale (5 folds)")
print(f"Booster : {best_booster_test.upper()}")
print("=" * 60)

cv_outer = StratifiedKFold(n_splits=N_CV, shuffle=True,
                             random_state=SEED)

cv_params = best_results[best_booster_test]['best_params'].copy()
cv_params.update({
    'booster'           : best_booster_test,
    'scale_pos_weight'  : scale_pos_weight,
    'random_state'      : SEED,
    'verbosity'         : 0,
    'eval_metric'       : 'auc',
    'use_label_encoder' : False,
})

cv_scores  = []
cv_details = []

for fold, (tr_idx, val_idx) in enumerate(
    cv_outer.split(X_static_v2, y_static_v2)
):
    X_f_tr, X_f_val = X_static_v2[tr_idx], X_static_v2[val_idx]
    y_f_tr, y_f_val = y_static_v2[tr_idx],  y_static_v2[val_idx]

    spw = (y_f_tr == 0).sum() / (y_f_tr == 1).sum()
    cv_params_f = cv_params.copy()
    cv_params_f['scale_pos_weight'] = spw

    model_cv = xgb.XGBClassifier(**cv_params_f)
    model_cv.fit(X_f_tr, y_f_tr, verbose=False)

    proba_cv = model_cv.predict_proba(X_f_val)[:, 1]
    label_cv = (proba_cv >= 0.5).astype(int)

    fold_auc = roc_auc_score(y_f_val, proba_cv)
    fold_ap  = average_precision_score(y_f_val, proba_cv)

    tp_f = ((label_cv==1) & (y_f_val==1)).sum()
    fp_f = ((label_cv==1) & (y_f_val==0)).sum()
    fn_f = ((label_cv==0) & (y_f_val==1)).sum()
    prec_f = tp_f / (tp_f + fp_f + 1e-8)
    rec_f  = tp_f / (tp_f + fn_f + 1e-8)
    f1_f   = 2*prec_f*rec_f / (prec_f+rec_f+1e-8)

    cv_scores.append(fold_auc)
    cv_details.append({
        'Fold'    : fold + 1,
        'AUC'     : fold_auc,
        'AP'      : fold_ap,
        'Precision': prec_f,
        'Recall'  : rec_f,
        'F1'      : f1_f,
    })
    print(f"  Fold {fold+1} | AUC={fold_auc:.4f} | "
          f"AP={fold_ap:.4f} | F1={f1_f:.4f}")

cv_df = pd.DataFrame(cv_details)
print(f"\nRésumé CV ({best_booster_test.upper()}) :")
print(f"  AUC : {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"  AP  : {cv_df['AP'].mean():.4f} ± {cv_df['AP'].std():.4f}")
print(f"  F1  : {cv_df['F1'].mean():.4f} ± {cv_df['F1'].std():.4f}")

# Graphique CV
fig, ax = plt.subplots(figsize=(10, 5))
metrics_cv = ['AUC', 'AP', 'Precision', 'Recall', 'F1']
colors_cv  = ['#7C3AED','#2563EB','#16A34A','#D97706','#DC2626']
x     = np.arange(N_CV)
width = 0.15

for j, (metric, color) in enumerate(zip(metrics_cv, colors_cv)):
    ax.bar(x + j*width, cv_df[metric].tolist(), width,
           label=metric, color=color, alpha=0.85)

ax.axhline(y=np.mean(cv_scores), color='black', ls='--',
           alpha=0.5,
           label=f'AUC mean={np.mean(cv_scores):.4f}')
ax.set_xticks(x + width*2)
ax.set_xticklabels([f'Fold {i+1}' for i in range(N_CV)])
ax.set_ylabel('Score')
ax.set_title(f'Cross-Validation — XGBoost {best_booster_test.upper()} + Optuna',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# ETAPE 7 — SHAP SUR LE MEILLEUR MODELE
# ════════════════════════════════════════════════════════════
print("\nÉTAPE 7 — SHAP")
print("=" * 60)

readable_names = [
    feature_names_readable.get(f, f)
    for f in static_features_v2
]

explainer   = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X_te)

if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

# Summary plot
plt.figure(figsize=(11, 7))
shap.summary_plot(shap_vals, X_te,
                  feature_names=readable_names,
                  show=False, max_display=15)
plt.title(f'SHAP Summary — XGBoost {best_booster_test.upper()} Jour 0',
          fontweight='bold')
plt.tight_layout()
plt.show()

# Bar plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_te,
                  feature_names=readable_names,
                  plot_type='bar', show=False, max_display=15)
plt.title(f'SHAP Feature Importance — '
          f'XGBoost {best_booster_test.upper()}',
          fontweight='bold')
plt.tight_layout()
plt.show()

# Waterfall
for test_idx, label in [
    (np.where(y_te == 1)[0][0], 'À Risque'),
    (np.where(y_te == 0)[0][0], 'Non À Risque'),
]:
    shap_exp = shap.Explanation(
        values        = shap_vals[test_idx],
        base_values   = explainer.expected_value,
        data          = X_te[test_idx],
        feature_names = readable_names
    )
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(shap_exp, show=False, max_display=12)
    plt.title(f'SHAP Waterfall — {label}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# RESUME FINAL
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("RÉSUMÉ FINAL")
print("=" * 60)

print(f"\nComparaison gbtree vs dart :")
display(comparison_df.round(4))

print(f"\nBooster retenu : {best_booster_test.upper()}")
print(f"  CV AUC  : {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"  Test AUC: {auc:.4f}")
print(f"  Test F1 : {f1:.4f}")
print(f"  Brier   : {brier:.4f}")

mean_abs = np.abs(shap_vals).mean(axis=0)
top3_idx = np.argsort(mean_abs)[::-1][:3]
print(f"\nTop 3 features SHAP :")
for i in top3_idx:
    print(f"  {readable_names[i]:<35} SHAP={mean_abs[i]:.4f}")

print(f"\n Modèle statique Jour 0 finalisé")